<a href="https://colab.research.google.com/github/VJN1089/CNS/blob/main/Copy_of_PythonProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install gradio pandas matplotlib

In [2]:
import sqlite3
import hashlib
import secrets
import re
import json
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from pathlib import Path

DB_PATH = "/content/spendwise.db"

CATEGORIES = [
    "Food", "Transport", "Shopping", "Bills",
    "Entertainment", "Health", "Education", "Other"
]

DEFAULT_BUDGETS = {
    "Food": 5000,
    "Transport": 2500,
    "Shopping": 4000,
    "Bills": 3500,
    "Entertainment": 2000,
    "Health": 2500,
    "Education": 2000,
    "Other": 3000
}

def connect_db():
    return sqlite3.connect(DB_PATH)

def initialize_database():
    with connect_db() as con:
        con.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                salt TEXT NOT NULL,
                password_hash TEXT NOT NULL
            )
        """)

        con.execute("""
            CREATE TABLE IF NOT EXISTS transactions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id INTEGER NOT NULL,
                date TEXT NOT NULL,
                description TEXT NOT NULL,
                amount REAL NOT NULL CHECK(amount > 0),
                category TEXT NOT NULL,
                payment_method TEXT DEFAULT 'Unknown',
                FOREIGN KEY(user_id) REFERENCES users(id)
            )
        """)

        con.execute("""
            CREATE TABLE IF NOT EXISTS budgets (
                user_id INTEGER NOT NULL,
                category TEXT NOT NULL,
                monthly_limit REAL NOT NULL,
                PRIMARY KEY(user_id, category)
            )
        """)

def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        bytes.fromhex(salt),
        200000
    ).hex()

initialize_database()

print("SpendWise database initialized successfully.")
print("Tables ready: users, transactions, budgets.")

SpendWise database initialized successfully.
Tables ready: users, transactions, budgets.


In [3]:
CATEGORY_RULES = {
    "Food": r"\b(food|restaurant|cafe|coffee|swiggy|zomato|grocery|pizza|bakery)\b",
    "Transport": r"\b(uber|ola|taxi|metro|bus|train|fuel|petrol|parking|rapido)\b",
    "Shopping": r"\b(amazon|flipkart|mall|clothing|shirt|shoes|shopping|retail)\b",
    "Bills": r"\b(electricity|water bill|internet|wifi|recharge|broadband|rent|gas bill)\b",
    "Entertainment": r"\b(movie|cinema|netflix|spotify|game|concert|theatre)\b",
    "Health": r"\b(pharmacy|hospital|doctor|clinic|medicine|medical)\b",
    "Education": r"\b(book|course|tuition|college|udemy|exam fee|stationery)\b"
}

def categorize_expense(description):
    description = str(description).lower()

    for category, pattern in CATEGORY_RULES.items():
        if re.search(pattern, description, re.IGNORECASE):
            return category

    return "Other"

# Test the categorization rules
examples = [
    "Zomato dinner",
    "Uber ride",
    "Amazon shopping",
    "Electricity bill",
    "Netflix subscription",
    "Pharmacy medicine",
    "College stationery"
]

for item in examples:
    print(f"{item:25} -> {categorize_expense(item)}")

Zomato dinner             -> Food
Uber ride                 -> Transport
Amazon shopping           -> Shopping
Electricity bill          -> Bills
Netflix subscription      -> Entertainment
Pharmacy medicine         -> Health
College stationery        -> Education


In [4]:
import os
import hmac
from datetime import date

# ---------- USER REGISTRATION AND LOGIN ----------

def register_user(username, password, confirm_password):
    username = (username or "").strip().lower()

    if not re.fullmatch(r"[a-z0-9_]{3,30}", username):
        return "Username must be 3–30 characters: letters, numbers or underscore."

    if not password or len(password) < 8:
        return "Password must contain at least 8 characters."

    if password != confirm_password:
        return "Passwords do not match."

    salt = secrets.token_hex(16)
    password_hash = hash_password(password, salt)

    try:
        with connect_db() as con:
            cur = con.execute(
                "INSERT INTO users(username, salt, password_hash) VALUES (?, ?, ?)",
                (username, salt, password_hash)
            )
            user_id = cur.lastrowid

            for category, amount in DEFAULT_BUDGETS.items():
                con.execute(
                    "INSERT INTO budgets(user_id, category, monthly_limit) VALUES (?, ?, ?)",
                    (user_id, category, amount)
                )

        return "Account created successfully. You can now log in."

    except sqlite3.IntegrityError:
        return "That username already exists. Please choose another."

def login_user(username, password):
    username = (username or "").strip().lower()

    with connect_db() as con:
        row = con.execute(
            "SELECT id, salt, password_hash FROM users WHERE username = ?",
            (username,)
        ).fetchone()

    if not row or not password:
        return None, "Invalid username or password."

    user_id, salt, stored_hash = row
    supplied_hash = hash_password(password, salt)

    if not hmac.compare_digest(supplied_hash, stored_hash):
        return None, "Invalid username or password."

    return user_id, f"Welcome to SpendWise, {username}!"

# ---------- CSV IMPORT ----------

def import_transactions(user_id, uploaded_file):
    if not user_id:
        return "Please log in first."

    if uploaded_file is None:
        return "Please upload a CSV file."

    path = uploaded_file if isinstance(uploaded_file, str) else uploaded_file.name

    try:
        df = pd.read_csv(path)
        df.columns = [
            str(c).strip().lower().replace(" ", "_")
            for c in df.columns
        ]

        aliases = {
            "transaction_date": "date",
            "merchant": "description",
            "details": "description",
            "expense": "amount",
            "value": "amount"
        }
        df = df.rename(columns=aliases)

        required = {"date", "description", "amount"}
        if not required.issubset(df.columns):
            return "CSV needs these columns: date, description, amount."

        df["date"] = pd.to_datetime(
            df["date"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")

        df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
        df["description"] = df["description"].fillna("").astype(str).str.strip()

        df = df.dropna(subset=["date", "amount"])
        df = df[
            (df["description"] != "") &
            (df["amount"] > 0)
        ].copy()

        if df.empty:
            return "No valid transactions found. Check dates and positive amounts."

        if "category" not in df.columns:
            df["category"] = df["description"].apply(categorize_expense)
        else:
            df["category"] = df["category"].fillna("").astype(str).str.strip()
            df.loc[df["category"] == "", "category"] = (
                df.loc[df["category"] == "", "description"]
                .apply(categorize_expense)
            )
            df.loc[~df["category"].isin(CATEGORIES), "category"] = (
                df.loc[~df["category"].isin(CATEGORIES), "description"]
                .apply(categorize_expense)
            )

        if "payment_method" not in df.columns:
            df["payment_method"] = "Unknown"

        df["payment_method"] = (
            df["payment_method"].fillna("Unknown").astype(str)
        )

        with connect_db() as con:
            con.executemany("""
                INSERT INTO transactions
                (user_id, date, description, amount, category, payment_method)
                VALUES (?, ?, ?, ?, ?, ?)
            """, [
                (
                    int(user_id),
                    row.date,
                    row.description,
                    float(row.amount),
                    row.category,
                    row.payment_method
                )
                for row in df.itertuples(index=False)
            ])

        return f"Successfully imported {len(df)} transactions."

    except Exception as e:
        return f"Could not import CSV: {e}"

# ---------- READ USER DATA ----------

def get_transactions(user_id):
    if not user_id:
        return pd.DataFrame(
            columns=["date", "description", "amount", "category", "payment_method"]
        )

    with connect_db() as con:
        df = pd.read_sql_query("""
            SELECT date, description, amount, category, payment_method
            FROM transactions
            WHERE user_id = ?
            ORDER BY date DESC, id DESC
        """, con, params=(int(user_id),))

    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    return df

def get_budgets(user_id):
    if not user_id:
        return DEFAULT_BUDGETS.copy()

    with connect_db() as con:
        rows = con.execute("""
            SELECT category, monthly_limit
            FROM budgets WHERE user_id = ?
        """, (int(user_id),)).fetchall()

    return dict(rows)

def save_budget(user_id, category, amount):
    if not user_id:
        return "Please log in first."

    if category not in CATEGORIES or amount is None or float(amount) < 0:
        return "Select a valid category and enter a non-negative budget."

    with connect_db() as con:
        con.execute("""
            INSERT INTO budgets(user_id, category, monthly_limit)
            VALUES (?, ?, ?)
            ON CONFLICT(user_id, category)
            DO UPDATE SET monthly_limit = excluded.monthly_limit
        """, (int(user_id), category, float(amount)))

    return f"Monthly budget for {category} saved: ₹{float(amount):,.2f}"

# ---------- MONTHLY DASHBOARD AND ALERTS ----------

def show_dashboard(user_id, selected_month=None):
    df = get_transactions(user_id)

    if df.empty:
        return (
            "## SpendWise Dashboard\n\nNo transactions yet. Upload a CSV to begin.",
            pd.DataFrame(),
            None,
            pd.DataFrame()
        )

    df = df.dropna(subset=["date"]).copy()
    df["month"] = df["date"].dt.strftime("%Y-%m")

    months = sorted(df["month"].unique())

    if selected_month not in months:
        selected_month = months[-1]

    monthly = df[df["month"] == selected_month].copy()
    total = float(monthly["amount"].sum())
    count = len(monthly)
    average = total / count if count else 0
    budgets = get_budgets(user_id)

    summary = f"""
# 💸 SpendWise Dashboard

**Month:** {selected_month}

| Metric | Result |
|---|---:|
| Total spending | ₹{total:,.2f} |
| Transactions | {count} |
| Average transaction | ₹{average:,.2f} |
"""

    category_totals = (
        monthly.groupby("category", as_index=False)["amount"]
        .sum().sort_values("amount", ascending=False)
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    if not category_totals.empty:
        ax.bar(category_totals["category"], category_totals["amount"])
    ax.set_title(f"Spending by Category — {selected_month}")
    ax.set_ylabel("Amount (₹)")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()

    alerts = []

    for category in CATEGORIES:
        spent = float(
            monthly.loc[monthly["category"] == category, "amount"].sum()
        )
        limit = float(budgets.get(category, 0))
        percentage = spent / limit * 100 if limit > 0 else 0

        if limit > 0 and spent > limit:
            status = "OVER BUDGET"
        elif limit > 0 and percentage >= 80:
            status = "Warning: 80%+ used"
        else:
            status = "Within budget"

        alerts.append({
            "Category": category,
            "Spent (₹)": round(spent, 2),
            "Budget (₹)": round(limit, 2),
            "Budget Used (%)": round(percentage, 1),
            "Status": status
        })

    alert_df = pd.DataFrame(alerts)

    if (alert_df["Status"] == "OVER BUDGET").any():
        summary += "\n\n🚨 **Alert:** At least one category is over budget."
    elif (alert_df["Status"] == "Warning: 80%+ used").any():
        summary += "\n\n⚠️ **Warning:** At least one category is close to its limit."
    else:
        summary += "\n\n✅ No budget warnings for this month."

    display_df = monthly[
        ["date", "description", "amount", "category", "payment_method"]
    ].copy()
    display_df["date"] = display_df["date"].dt.strftime("%Y-%m-%d")

    return summary, display_df, fig, alert_df

# ---------- EXPORT CSV AND JSON ----------

def export_report(user_id, selected_month, report_format):
    if not user_id:
        return None, "Please log in first."

    df = get_transactions(user_id)

    if df.empty:
        return None, "No transactions to export."

    df["month"] = df["date"].dt.strftime("%Y-%m")
    if selected_month not in df["month"].unique():
        return None, "No transactions for the selected month."

    df = df[df["month"] == selected_month].drop(columns=["month"]).copy()
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

    output_path = f"/content/spendwise_report_{int(user_id)}_{selected_month}.{report_format}"

    if report_format == "csv":
        df.to_csv(output_path, index=False)
    else:
        df.to_json(output_path, orient="records", indent=2)

    return output_path, "Report prepared successfully."

# ---------- GRADIO USER INTERFACE ----------

with gr.Blocks(title="SpendWise", theme=gr.themes.Soft()) as app:
    user_state = gr.State(None)

    gr.Markdown("# 💸 SpendWise")
    gr.Markdown("### Automated Expense Categorization and Budget Alert System")

    with gr.Tab("Register"):
        reg_username = gr.Textbox(label="Choose username")
        reg_password = gr.Textbox(label="Password", type="password")
        reg_confirm = gr.Textbox(label="Confirm password", type="password")
        reg_button = gr.Button("Create Account")
        reg_status = gr.Markdown()

        reg_button.click(
            register_user,
            inputs=[reg_username, reg_password, reg_confirm],
            outputs=reg_status
        )

    with gr.Tab("Login"):
        login_username = gr.Textbox(label="Username")
        login_password = gr.Textbox(label="Password", type="password")
        login_button = gr.Button("Login", variant="primary")
        login_status = gr.Markdown()

        login_button.click(
            login_user,
            inputs=[login_username, login_password],
            outputs=[user_state, login_status]
        )

    with gr.Tab("Upload CSV"):
        gr.Markdown(
            "Required CSV columns: `date`, `description`, `amount`. "
            "Optional columns: `category`, `payment_method`."
        )
        upload_file = gr.File(label="Choose transaction CSV", file_types=[".csv"], type="filepath")
        upload_button = gr.Button("Import Transactions")
        upload_status = gr.Markdown()

        upload_button.click(
            import_transactions,
            inputs=[user_state, upload_file],
            outputs=upload_status
        )

    with gr.Tab("Dashboard"):
        month_choice = gr.Textbox(
            label="Reporting month (YYYY-MM)",
            value=date.today().strftime("%Y-%m"),
            placeholder="Example: 2026-10"
        )
        dashboard_button = gr.Button("Refresh Dashboard")
        dashboard_summary = gr.Markdown()
        transaction_table = gr.Dataframe(label="Transactions", interactive=False)
        spending_chart = gr.Plot(label="Spending Chart")
        budget_alerts = gr.Dataframe(label="Budget Status", interactive=False)

        dashboard_button.click(
            show_dashboard,
            inputs=[user_state, month_choice],
            outputs=[
                dashboard_summary,
                transaction_table,
                spending_chart,
                budget_alerts
            ]
        )

    with gr.Tab("Monthly Budgets"):
        budget_category = gr.Dropdown(
            choices=CATEGORIES,
            value="Food",
            label="Expense category"
        )
        budget_amount = gr.Number(
            value=5000,
            minimum=0,
            label="Monthly spending limit (₹)"
        )
        budget_button = gr.Button("Save Budget")
        budget_status = gr.Markdown()

        budget_button.click(
            save_budget,
            inputs=[user_state, budget_category, budget_amount],
            outputs=budget_status
        )

    with gr.Tab("Export Reports"):
        export_month = gr.Textbox(
            label="Month (YYYY-MM)",
            value=date.today().strftime("%Y-%m")
        )
        export_format = gr.Radio(
            choices=["csv", "json"],
            value="csv",
            label="Report format"
        )
        export_button = gr.Button("Generate Report")
        export_file = gr.File(label="Download report")
        export_status = gr.Markdown()

        export_button.click(
            export_report,
            inputs=[user_state, export_month, export_format],
            outputs=[export_file, export_status]
        )

app.launch(share=True)

/tmp/ipykernel_1979/1088847418.py:321: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="SpendWise", theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e84dd41f6c902d15d5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
